**Source:** [Colab Link](https://colab.research.google.com/drive/1vmsgpANP1ZUFeWX6YaKn3qAzkRB97qgD?usp=sharing)

# **1. Environment Setup**
This section handles the installation of all necessary dependencies required to run the candidate ranking pipeline.

* **pandas & numpy:** For efficient data manipulation and numerical operations.

* **faiss-cpu:** Used for high-performance vector similarity search.

* **sentence-transformers:** To load the **bge-base-en-v1.5** embedding model for feature extraction.

* **python-docx:** Required for parsing the job description files.

* **pyarrow:** For optimized reading/writing of Parquet data artifacts.

In [1]:
!pip install pandas numpy faiss-cpu sentence-transformers python-docx pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 15.2 MB/s eta 0:00:00



# **2. Environment Initialization**
This cell configures the foundational components of the ranking pipeline:

* **Dependency Imports:** Loads essential libraries for data processing (pandas, numpy), semantic search (faiss), and text extraction (docx).

* **Embedding Model:** Initializes the BAAI/bge-base-en-v1.5 model, which acts as the core engine for converting candidate text and job requirements into high-dimensional vector space.



In [2]:
import json
import os
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import re
from tqdm import tqdm
import time
import faiss
import docx
import textwrap
import hashlib

model = SentenceTransformer("BAAI/bge-base-en-v1.5")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# **3. Data Ingestion & Configuration**
This cell automatically prepares the environment by fetching the required artifacts and test data.

* **Default Behavior:** If no custom files are found, the system automatically pulls the verified `sample candidates.jsonl` and benchmark `job_description.docx` from the GitHub repository.

* **Custom Candidate Testing:** If you wish to test the system with your own custom candidate list, click the Folder icon in the left sidebar, drag and drop your **`candidates.jsonl`** file into the directory, and re-run this cell. The system will prioritize your uploaded file.

* **Benchmarking Policy:** The included `job_description.docx` serves as the static benchmark to ensure consistent evaluation metrics.

* **Advanced Configuration:** To evaluate a custom Job Description, you must generate a corresponding `jd_capability_vector.json`. This is achieved by performing an LLM-based API call at the preprocessing stage to map your specific job requirements to the `universal taxonomy` with appropriate weights.

In [3]:
import json
import docx
import os
import requests

def load_data():
    # File paths
    cands_input = "candidates.jsonl"
    jd_input = "job_description.docx"

    # 1. Handle Candidates (Check if user uploaded a custom one)
    if os.path.exists(cands_input):
        print("[+] Custom candidate file detected. Using local 'candidates.jsonl'.")
    else:
        print("[*] No custom candidates file detected. Downloading sample data...")
        url = "https://raw.githubusercontent.com/Saswata-pal/HireWise/main/assets/sample_100_candidates.jsonl"
        r = requests.get(url)
        with open(cands_input, 'wb') as f:
            f.write(r.content)

    # 2. Handle JD (Always use the benchmark version)
    if not os.path.exists(jd_input):
        print("[*] Downloading benchmark Job Description...")
        url = "https://raw.githubusercontent.com/Saswata-pal/HireWise/main/assets/job_description.docx"
        r = requests.get(url)
        with open(jd_input, 'wb') as f:
            f.write(r.content)
    else:
        print("[+] Benchmark Job Description verified.")

    # 3. Load Candidates
    candidates = []
    with open(cands_input, "r") as f:
        for line in f:
            if line.strip():
                candidates.append(json.loads(line))

    # 4. Load JD Text
    doc = docx.Document(jd_input)
    jd_text = "\n".join([para.text for para in doc.paragraphs])

    return candidates, jd_text

# Run Ingestion
candidates, jd_text = load_data()
print(f"[+] Sandbox ready. Loaded {len(candidates)} candidates.")
print(f"[*] Job Description loaded ({len(jd_text)} characters).")

[*] No custom candidates file detected. Downloading sample data...
[*] Downloading benchmark Job Description...
[+] Sandbox ready. Loaded 100 candidates.
[*] Job Description loaded (9572 characters).


# **4. Taxonomy Configuration**
This cell initializes the `UNIVERSAL_TECH_TAXONOMY` schema, mapping candidates and job descriptions into a shared high-dimensional coordinate system. This ensures that the retrieval engine operates on a standardized, domain-aware logic from preprocessing to final ranking.

> **⚠️ Domain Specificity:** This taxonomy is strictly calibrated for `Software Engineering`. Using it for other industries (e.g., Legal, Finance) will cause semantic drift, resulting in inaccurate rankings. You must update these dimensions if the target domain changes.

In [4]:
UNIVERSAL_TECH_TAXONOMY = [
    # --- 1. CORE AI, ML & DATA SCIENCE ---
    "machine_learning_model_training", "deep_learning_neural_networks",
    "large_language_models_and_generative_ai", "llm_fine_tuning_and_alignment",
    "rag_retrieval_augmented_generation", "vector_search_and_embedding_infrastructure",
    "semantic_search_and_dense_retrieval", "model_quantization_and_inference_optimization",
    "distributed_gpu_training", "nlp_text_processing_and_transformers",
    "computer_vision_and_image_processing", "speech_recognition_and_audio_processing",
    "time_series_forecasting_and_anomaly_detection", "reinforcement_learning_systems",
    "mlops_and_model_deployment", "feature_store_engineering",
    "search_relevance_and_information_retrieval", "learning_to_rank_models",
    "recommendation_systems", "hybrid_search_retrieval_systems",
    "ml_observability_and_monitoring",

    # --- 2. DATA ENGINEERING & PIPELINES ---
    "real_time_stream_processing", "batch_data_processing",
    "data_pipeline_orchestration", "data_warehouse_architecture",
    "data_lakehouse_architecture", "relational_data_modeling_and_schema_design",
    "graph_database_architecture", "olap_analytical_query_optimization",
    "data_governance_and_data_quality", "business_intelligence_and_analytics",

    # --- 3. BACKEND & DISTRIBUTED SYSTEMS ---
    "microservices_architecture", "monolithic_architecture_refactoring",
    "api_design_and_development", "event_driven_architecture_and_message_brokers",
    "serverless_architecture", "distributed_consensus_and_state_machine_replication",
    "backend_performance_profiling_and_tuning", "low_level_memory_management",
    "concurrent_and_parallel_programming", "websocket_and_real_time_communications",

    # --- 4. DATABASE & STORAGE OPS ---
    "relational_database_administration", "nosql_database_design",
    "distributed_caching_layers", "database_sharding_and_replication_strategies",
    "query_optimization_and_index_tuning", "acid_transactions_and_isolation_levels",
    "distributed_storage_systems",

    # --- 5. FRONTEND & CLIENT-SIDE ---
    "component_based_ui_frameworks", "frontend_state_management",
    "server_side_rendering_and_hydration", "single_page_application_spa_architecture",
    "css_architecture_and_styling", "web_accessibility_compliance",
    "frontend_build_tooling", "webassembly_performance_optimization",
    "ui_ux_animation_and_microinteractions", "browser_rendering_optimization",

    # --- 6. MOBILE APP DEVELOPMENT ---
    "native_ios_development", "native_android_development",
    "cross_platform_mobile_development", "mobile_app_state_and_offline_storage",
    "mobile_ui_performance_profiling", "app_store_deployment_and_ci_cd",

    # --- 7. CLOUD PLATFORMS & INFRASTRUCTURE ---
    "public_cloud_infrastructure", "multi_cloud_and_hybrid_architecture",
    "infrastructure_as_code", "containerization",
    "container_orchestration", "cloud_networking_and_load_balancing",
    "server_provisioning_and_configuration",

    # --- 8. DEVOPS, SRE & OBSERVABILITY ---
    "ci_cd_pipeline_automation", "system_observability_and_metrics",
    "distributed_tracing", "log_aggregation_and_analysis",
    "incident_response_and_on_call_management", "site_reliability_engineering",
    "linux_system_administration_and_kernel_tuning", "chaos_engineering_and_fault_tolerance",
    "experimentation_and_ab_testing",

    # --- 9. SECURITY & COMPLIANCE ---
    "web_application_security", "identity_and_access_management",
    "cryptography_and_encryption_in_transit", "network_security_and_defense",
    "penetration_testing_and_vulnerability_scanning", "security_compliance_and_governance",
    "secure_software_development_lifecycle",

    # --- 10. LOW-LEVEL, FIRMWARE & GAME DEV ---
    "embedded_systems_and_firmware", "real_time_operating_systems_rtos",
    "game_engine_architecture", "graphics_programming",
    "physics_simulation_and_rendering", "iot_device_communication_protocols",

    # --- 11. SOFTWARE ENGINEERING PRACTICES & LEADERSHIP ---
    "test_driven_development_and_testing", "end_to_end_integration_qa_automation",
    "agile_engineering_leadership", "cross_functional_team_mentorship_and_management",
    "system_architecture_and_design", "technical_debt_management_and_refactoring",
    "open_source_contributions_and_maintainership", "product_engineering_and_feature_ownership",

    # --- 12. SPECIALIZED DOMAINS ---
    "payment_systems_and_fintech", "blockchain_and_web3_infrastructure"
]

print(f"[*] Initialized Taxonomy with {len(UNIVERSAL_TECH_TAXONOMY)} dimensions.")
with open("taxonomy_schema.json", "w") as f:
  json.dump(UNIVERSAL_TECH_TAXONOMY, f, indent=4)

[*] Initialized Taxonomy with 103 dimensions.


# **5. Trust Evaluation Engine**
This module functions as an automated anti-fraud and credibility layer, applying a dual-gating mechanism to validate candidate profiles.

* **Hard Disqualification (`fatal`):** Instantly filters out profiles containing logical impossibilities (e.g., 4+ concurrent roles, negative job durations).

* **Weighted Trust Degradation (`penalty`):** Rather than rejecting borderline candidates, the engine applies proportional penalties to the `trust_multiplier` (ranging from 1.0 down to 0.3) based on behavioral red flags.

* **Dynamic Constraints:** This ensures that while a candidate might remain in the talent pool, their ranking is dynamically suppressed if they exhibit patterns of "Keyword Stuffing," "Title Chasing," or "Fake Prodigy" growth.

In [5]:
def evaluate_candidate_trust(candidate):

    profile = candidate.get("profile", {})
    history = candidate.get("career_history", [])
    skills = candidate.get("skills", [])

    current_title = profile.get("current_title", "").lower()
    claimed_yoe = profile.get("years_of_experience", 0)

    result = {
        "fatal": False,
        "trust_multiplier": 1.0,
        "risk_flags": []
    }

    penalty = 0.0


    expert_zero_count = sum(1 for s in skills if s.get("proficiency", "").lower() == "expert" and s.get("duration_months", 0) <= 1)
    if expert_zero_count >= 3:
        result["fatal"] = True
        return result

    for s in skills:
        if s.get("duration_months", 0) > (claimed_yoe * 12) + 60:
            penalty += 0.10
            result["risk_flags"].append("Exaggerated Skill Duration")
            break

    for job in history:
        duration = job.get("duration_months")
        if duration is not None:
            if duration < 0:
                result["fatal"] = True
                return result
            if duration > (claimed_yoe * 12) + 36:
                result["fatal"] = True
                return result

    events = []
    for job in history:
        start_str = job.get("start_date")
        end_str = job.get("end_date")
        duration_m = job.get("duration_months", 0)
        if not start_str: continue
        try:
            start_dt = datetime.strptime(start_str, "%Y-%m-%d")
            if end_str:
                end_dt = datetime.strptime(end_str, "%Y-%m-%d")
            else:
                end_dt = start_dt + timedelta(days=int(duration_m * 30.4375))

            events.append((start_dt, 1))
            events.append((end_dt, -1))
        except ValueError: continue

    events.sort(key=lambda x: (x[0], x[1]))
    current_concurrent, max_concurrent = 0, 0
    for date, delta in events:
        current_concurrent += delta
        max_concurrent = max(max_concurrent, current_concurrent)

    if max_concurrent >= 4:
        result["fatal"] = True
        return result

    if max_concurrent == 3:
        penalty += 0.15
        result["risk_flags"].append("High Concurrent Employment")

    valid_start_dates, valid_end_dates = [], []
    for job in history:
        start_str = job.get("start_date")
        end_str = job.get("end_date")
        duration_m = job.get("duration_months", 0)
        try:
            if start_str:
                start_dt = datetime.strptime(start_str, "%Y-%m-%d")
                valid_start_dates.append(start_dt)

                if end_str:
                    valid_end_dates.append(datetime.strptime(end_str, "%Y-%m-%d"))
                else:
                    valid_end_dates.append(start_dt + timedelta(days=int(duration_m * 30.4375)))
        except ValueError: continue

    if valid_start_dates and valid_end_dates:
        timeline_years = (max(valid_end_dates) - min(valid_start_dates)).days / 365.25
        if timeline_years > claimed_yoe + 4:
            penalty += 0.10
            result["risk_flags"].append("Significant Career Gap / YOE Mismatch")

    for s in skills:
        if s.get("proficiency", "").lower() == "expert" and s.get("duration_months", 0) < 12:
            penalty += 0.10
            result["risk_flags"].append("Premature Expert Claims")
            break

    advanced_count = sum(1 for s in skills if s.get("proficiency", "").lower() in ["advanced", "expert"])
    if claimed_yoe < 3 and advanced_count >= 15:
        penalty += 0.05
        result["risk_flags"].append("Unlikely Skill Inflation")

    non_tech_titles = ["accountant", "hr", "marketing", "sales", "recruiter", "finance", "admin"]
    if any(t in current_title for t in non_tech_titles) and not any(any(kw in str(j.get("title", "")).lower() for kw in ["engineer", "developer", "data"]) for j in history):
        if sum(1 for s in skills if s.get("name", "").lower() in ["rag", "kubernetes", "llm", "pinecone"]) >= 3:
            penalty += 0.20
            result["risk_flags"].append("Keyword Stuffing (Non-Tech History)")

    consulting_firms = ["tcs", "infosys", "wipro", "accenture", "cognizant", "capgemini"]
    if any(any(f in str(j.get("company", "")).lower() for f in consulting_firms) for j in history) and not any(not any(f in str(j.get("company", "")).lower() for f in consulting_firms) for j in history):
        penalty += 0.10
        result["risk_flags"].append("Pure Consulting Background")

    if len(history) >= 3 and sum(1 for j in history if j.get("duration_months", 999) <= 18) >= 3 and sum(1 for j in history if any(t in str(j.get("title", "")).lower() for t in ["senior", "staff", "principal"])) >= 2:
        penalty += 0.10
        result["risk_flags"].append("High Velocity Title Chasing")

    title_ranks = {"intern": 1, "junior": 2, "engineer": 3, "senior": 4, "lead": 5, "staff": 6, "principal": 7}
    if len(history) >= 2:
        sorted_history = sorted([j for j in history if j.get("start_date")], key=lambda x: x.get("start_date", ""))
        for i in range(len(sorted_history) - 1):
            old_r = next((r for kw, r in title_ranks.items() if kw in str(sorted_history[i].get("title", "")).lower()), None)
            new_r = next((r for kw, r in title_ranks.items() if kw in str(sorted_history[i+1].get("title", "")).lower()), None)
            if old_r and new_r and (new_r - old_r >= 5) and claimed_yoe < 5:
                penalty += 0.10
                result["risk_flags"].append("Unlikely Title Jump")
                break

    result["trust_multiplier"] = max(0.30, 1.0 - penalty)
    return result

# **6. GPU Batching & Deep Feature Extraction**
This pipeline stage performs the heavy-duty offline processing required to transform unstructured candidate profiles into a structured, queryable feature store.

* **High-Throughput Encoding:** Utilizes GPU-accelerated batching to perform semantic encoding of candidate data, mapping it into the high-dimensional vector space defined by your taxonomy.

* **Evidence Extraction:** Beyond just calculating scores, this stage isolates specific "evidence quotes" from the candidate's history to support the taxonomy mappings, which are critical for the final transparency and confidence scoring.

* **Impact Scoring:** Implements pattern recognition to quantify achievements (e.g., impact on growth, scale, or metrics), normalizing these into a measurable `impact_score`.

* **Offline Artifact Serialization:** Persists the processed matrices to disk (`.parquet` and `.npy` files). This is a critical design choice: it allows the final Ranking Stage to operate as a lightweight, purely offline engine, ensuring it meets strict CPU and latency constraints during evaluation.

In [11]:
CURRENT_DATE = datetime(2026, 6, 17)

print("[*] Loading Taxonomy & Precomputing Vectors...")
with open("taxonomy_schema.json", "r") as f:
    TECH_TAXONOMY = json.load(f)

tax_embeddings = model.encode(TECH_TAXONOMY, normalize_embeddings=True)

candidates = []
with open("/content/drive/MyDrive/Colab Notebooks/candidates.jsonl", "r") as f:
    for line in f:
        if line.strip():
            candidates.append(json.loads(line))

metadata = []
raw_vectors = []
tax_score_matrix = []

print(f"[*] Processing {len(candidates)} candidates using GPU Batching...")
BATCH_SIZE = 1000

for i in tqdm(range(0, len(candidates), BATCH_SIZE), desc="Processing Batches"):
    batch = candidates[i : i + BATCH_SIZE]

    all_sentences = []
    candidate_boundaries = []
    current_idx = 0

    for cand in batch:
        profile = cand.get("profile", {})
        history = cand.get("career_history", [])

        headline = profile.get("headline", "")
        summary = profile.get("summary", "")
        history_desc = " ".join([h.get("description", "") for h in history])
        full_text = f"{headline}. {summary}. {history_desc}"

        sentences = re.split(r'(?<=[.!?]) +', full_text)
        sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
        if not sentences:
            sentences = ["No detailed text provided."]

        all_sentences.extend(sentences)
        candidate_boundaries.append((current_idx, current_idx + len(sentences)))
        current_idx += len(sentences)

    batch_sent_vecs = model.encode(all_sentences, batch_size=256, normalize_embeddings=True)

    for c_idx, cand in enumerate(batch):
        start, end = candidate_boundaries[c_idx]
        sent_vecs = batch_sent_vecs[start:end]

        sim_matrix = np.dot(sent_vecs, tax_embeddings.T)

        max_sim_scores = np.max(sim_matrix, axis=0)
        best_sentence_indices = np.argmax(sim_matrix, axis=0)

        supporting_counts = np.maximum(0, np.sum(sim_matrix > 0.65, axis=0) - 1)

        bonus_multiplier = 1.0 + np.clip(supporting_counts * 0.02, 0.0, 0.10)

        adjusted_tax_scores = np.clip(max_sim_scores * bonus_multiplier, 0.0, 1.0)

        cand_text_lower = " ".join(all_sentences[start:end]).lower()
        impact_matches = len(re.findall(r'(\d+%|\d+\s*(million|m|k|users|requests|tps)|reduced|increased|improved|saved)', cand_text_lower))
        impact_score = min(impact_matches / 10.0, 1.0)

        evidence_dict = {}
        for tax_idx in range(len(UNIVERSAL_TECH_TAXONOMY)):
            tag = UNIVERSAL_TECH_TAXONOMY[tax_idx]
            best_idx = best_sentence_indices[tax_idx]
            highest_score = max_sim_scores[tax_idx]

            if highest_score >= 0.65:
                evidence_dict[tag] = all_sentences[start + best_idx]
            else:
                evidence_dict[tag] = "NO_EVIDENCE"

        raw_vec = np.mean(sent_vecs, axis=0)
        raw_vec = raw_vec / np.linalg.norm(raw_vec)

        profile = cand.get("profile", {})
        redrob_signals = cand.get("redrob_signals", {})
        yoe = profile.get("years_of_experience", 0)

        last_active_str = redrob_signals.get("last_active_date")
        if last_active_str:
            try:
                days_since_active = (CURRENT_DATE - datetime.strptime(last_active_str, "%Y-%m-%d")).days
            except ValueError:
                days_since_active = 999
        else:
            days_since_active = 999

        trust_data = evaluate_candidate_trust(cand)

        metadata.append({
            "candidate_id": cand.get("candidate_id"),
            "is_fatal": trust_data["fatal"],
            "trust_multiplier": trust_data["trust_multiplier"],
            "risk_flags": json.dumps(trust_data["risk_flags"]),
            "years_of_experience": yoe,

            "days_since_active": days_since_active,
            "applications_submitted": redrob_signals.get("applications_submitted_30d", 0),
            "response_rate": redrob_signals.get("recruiter_response_rate", 0.0),
            "completeness_score": redrob_signals.get("profile_completeness_score", 50.0) / 100.0,
            "saved_by_recruiters": redrob_signals.get("saved_by_recruiters_30d", 0),
            "interview_completion": redrob_signals.get("interview_completion_rate", 1.0),
            "notice_period": redrob_signals.get("notice_period_days", 30),
            "github_score": redrob_signals.get("github_activity_score", -1),

            "impact_score": impact_score,
            "evidence_dict": json.dumps(evidence_dict)
        })

        raw_vectors.append(raw_vec)
        tax_score_matrix.append(adjusted_tax_scores)

raw_matrix = np.vstack(raw_vectors).astype(np.float32)
tax_matrix = np.vstack(tax_score_matrix).astype(np.float32)

print("\n[*] Normalizing taxonomy scores...")
tax_matrix_normalized = np.clip(tax_matrix, 0.0, 1.0)

print("[*] Saving optimized artifacts to disk...")
pd.DataFrame(metadata).to_parquet("candidate_metadata.parquet", index=False)
np.save("candidate_raw_matrix.npy", raw_matrix)
np.save("candidate_tax_matrix_norm.npy", tax_matrix_normalized)

print("[+] Offline Processing Complete. Feature store updated.")

[*] Loading Taxonomy & Precomputing Vectors...
[*] Processing 100000 candidates using GPU Batching...


Processing Batches: 100%|██████████| 100/100 [47:42<00:00, 28.63s/it]



[*] Normalizing taxonomy scores...
[*] Saving optimized artifacts to disk...
[+] Offline Processing Complete. Feature store updated.


# **7. Deterministic Benchmark Injection (Pre-Computed Weights)**
To adhere to the secure, offline evaluation environment, this module utilizes **pre-computed taxonomy weights** for the Job Description.

* **Security & Reproducibility:** By pre-computing the JD-to-taxonomy mapping using an LLM prior to the sandbox stage, we eliminate the need for live API keys or network access during the evaluation.

* **Deterministic Results:** This ensures that every test run is 100% reproducible and immune to external API latency or service failures.

* **Architecture:** The `jd_capability_vector.json` acts as the "Benchmark Configuration." In a production environment, this file is the output of your upstream preprocessing pipeline. For this submission, it is provided as a static artifact to optimize the evaluation runtime.

In [12]:
weights = {
    "min_yoe": 5,
    "max_yoe": 9,
    "vector_search_and_embedding_infrastructure": 1.0,
    "semantic_search_and_dense_retrieval": 1.0,
    "hybrid_search_retrieval_systems": 1.0,
    "search_relevance_and_information_retrieval": 1.0,
    "experimentation_and_ab_testing": 1.0,
    "product_engineering_and_feature_ownership": 0.9,
    "recommendation_systems": 0.9,
    "nlp_text_processing_and_transformers": 0.9,
    "rag_retrieval_augmented_generation": 0.8,
    "large_language_models_and_generative_ai": 0.8,
    "system_architecture_and_design": 0.8,
    "ml_observability_and_monitoring": 0.8,
    "learning_to_rank_models": 0.7,
    "llm_fine_tuning_and_alignment": 0.7,
    "mlops_and_model_deployment": 0.7,
    "machine_learning_model_training": 0.6,
    "backend_performance_profiling_and_tuning": 0.6,
    "cross_functional_team_mentorship_and_management": 0.6,
    "open_source_contributions_and_maintainership": 0.5,
    "distributed_gpu_training": 0.4,
    "api_design_and_development": 0.4,
    "data_pipeline_orchestration": 0.3,
    "microservices_architecture": 0.3,
    "public_cloud_infrastructure": 0.2,
    "computer_vision_and_image_processing": 0.0,
    "speech_recognition_and_audio_processing": 0.0,
    "component_based_ui_frameworks": 0.0,
    "native_ios_development": 0.0,
    "game_engine_architecture": 0.0,
    "blockchain_and_web3_infrastructure": 0.0
}

output_path = "jd_capability_vector.json"

with open(output_path, "w") as f:
    json.dump(weights, f, indent=4)

print(f"[+] Successfully created {output_path} with {len(weights)-2} taxonomy weights and YOE range.")

[+] Successfully created jd_capability_vector.json with 30 taxonomy weights and YOE range.


# **8. Hybrid Ranking Engine & Reasoning Generator**
This final processing stage synthesizes semantic proximity, taxonomy capability, and behavioral heuristics into a single, unified `final_score`. It transforms raw vectors into actionable talent intelligence.

* **Multi-Factor Scoring:** Orchestrates a weighted ensemble of Semantic Search (FAISS), Technical Taxonomy mapping, and Impact Scoring, adjusted by dynamic penalties for trust, experience mismatches, and professional flakiness.

* **Explainable AI (XAI):** Dynamically generates natural language reasoning for every candidate. This engine cites specific "evidence quotes" from the candidate's history and explains rank variance, providing transparency into the model's decision-making.

* **Deliverable Serialization:** Exports the final, production-ready `submission.csv` containing the candidate rankings and justifying rationale, completing the pipeline.

In [17]:
start_time = time.time()

print("[*] Loading offline universal artifacts...")
metadata_df = pd.read_parquet("/content/candidate_metadata.parquet")
raw_matrix = np.load("/content/candidate_raw_matrix.npy")
tax_matrix = np.load("/content/candidate_tax_matrix_norm.npy")

with open("taxonomy_schema.json", "r") as f:
    UNIVERSAL_TECH_TAXONOMY = json.load(f)

print("[*] Loading LLM Taxonomy Weights & Parsing Raw JD...")

try:
    with open("/content/jd_capability_vector.json", "r") as f:
        jd_llm_weights = json.load(f)
except FileNotFoundError:
    print("[!] Warning: jd_capability_vector.json not found. Defaulting to empty weights.")
    jd_llm_weights = {}

jd_tax_vector = np.array([jd_llm_weights.get(tag, 0.0) for tag in UNIVERSAL_TECH_TAXONOMY], dtype=np.float32)

if np.sum(jd_tax_vector) > 0:
    jd_tax_vector = jd_tax_vector / np.sum(jd_tax_vector)

doc = docx.Document("/content/job_description.docx")
jd_text = "\n".join([para.text for para in doc.paragraphs]).lower()
jd_raw_vector = model.encode([jd_text], normalize_embeddings=True).astype(np.float32)

target_yoe_min = jd_llm_weights.get("min_yoe", 5)
target_yoe_max = jd_llm_weights.get("max_yoe", 9)

print(f"[*] Dynamic YOE Target Extracted: {target_yoe_min} to {target_yoe_max} years")

TOP_N = 14
top_core_jd_indices = np.argsort(jd_tax_vector)[-TOP_N:][::-1]

print("[*] Executing FAISS Exact Search...")
index = faiss.IndexFlatIP(768)
index.add(raw_matrix)
K_RETRIEVE = 5000
semantic_scores, semantic_indices = index.search(jd_raw_vector, K_RETRIEVE)

top_indices = semantic_indices[0]
top_semantic_scores = semantic_scores[0]

subset_tax_matrix_raw = tax_matrix[top_indices]
subset_metadata_raw = metadata_df.iloc[top_indices].reset_index(drop=True)

print("[*] Purging Fatal Profiles to optimize compute...")
valid_mask = ~subset_metadata_raw['is_fatal'].values

subset_metadata = subset_metadata_raw[valid_mask].copy()
subset_tax_matrix = subset_tax_matrix_raw[valid_mask]

top_semantic_scores = top_semantic_scores[valid_mask]

print(f"[*] Remaining valid candidates for scoring: {len(subset_metadata)}")

print("[*] Applying Hybrid Math & Behavioral Penalties...")
tax_scores = np.dot(subset_tax_matrix, jd_tax_vector)
impact_scores = subset_metadata['impact_score'].values

yoe = subset_metadata['years_of_experience'].values
yoe_multiplier = np.ones_like(yoe, dtype=float)

yoe_multiplier[(yoe >= target_yoe_min) & (yoe <= target_yoe_max)] = 1.05

yoe_multiplier[(yoe >= target_yoe_min - 1) & (yoe < target_yoe_min)] = 0.90
yoe_multiplier[(yoe >= target_yoe_min - 2) & (yoe < target_yoe_min - 1)] = 0.75
yoe_multiplier[(yoe >= target_yoe_min - 3) & (yoe < target_yoe_min - 2)] = 0.50
yoe_multiplier[yoe < target_yoe_min - 3] = 0.25

over_exp_diff = np.maximum(0, yoe - (target_yoe_max + 2))
over_penalty = np.maximum(0.90, 1.0 - (over_exp_diff * 0.01))
yoe_multiplier = np.where(yoe > (target_yoe_max + 2), over_penalty, yoe_multiplier)

days_since_active = subset_metadata['days_since_active'].values
apps_submitted = subset_metadata['applications_submitted'].values
response_rate = subset_metadata['response_rate'].values
interview_comp = subset_metadata['interview_completion'].values
notice_period = subset_metadata['notice_period'].values
saved_by_rec = subset_metadata['saved_by_recruiters'].values
github_score = subset_metadata['github_score'].values

is_dead_profile = (days_since_active > 180) & (apps_submitted == 0)
activity_penalty = np.where(is_dead_profile, 0.85, 1.0)
response_penalty = np.where(response_rate < 0.10, 0.95, 1.0)
flakiness_penalty = np.where(interview_comp < 0.50, 0.85, 1.0)
notice_penalty = np.where(notice_period > 60, 0.97, 1.0)

market_boost = np.clip(saved_by_rec * 0.01, 0.0, 0.10)
github_boost = np.where(github_score > 80, 0.05, 0.0)

behavioral_modifier = activity_penalty * response_penalty * flakiness_penalty * notice_penalty * (1.0 + market_boost + github_boost)

base_score = (tax_scores * 0.32) + (top_semantic_scores * 0.48) + (impact_scores * 0.15) + 0.05
subset_metadata['final_score'] = base_score * yoe_multiplier * subset_metadata['trust_multiplier'] * behavioral_modifier
subset_metadata['final_score'] = subset_metadata['final_score'].round(4)
subset_metadata['tax_scores'] = list(subset_tax_matrix)

print("[*] Sorting Top Validators...")
clean_candidates = subset_metadata.sort_values(by=['final_score', 'candidate_id'], ascending=[False, True])
top_100 = clean_candidates.head(100).copy()

print("[*] Generating Step 16 Confidence Scores and Reasoning...")
submission = []

for rank, (idx, row) in enumerate(top_100.iterrows(), start=1):
    cand_tax = row['tax_scores']
    try:
        evidence = json.loads(row.get('evidence_dict', '{}'))
    except:
        evidence = {}

    yoe = row['years_of_experience']

    top_cand_scores = cand_tax[top_core_jd_indices]
    covered_count = np.sum(top_cand_scores >= 0.6)
    evidence_coverage = covered_count / float(TOP_N)

    evidence_strength = np.mean(top_cand_scores[top_cand_scores >= 0.6]) if covered_count > 0 else 0.0
    signal_consistency = max(0.0, 1.0 - np.std(top_cand_scores))
    profile_completeness = row['completeness_score']

    confidence = (0.40 * evidence_coverage) + \
                 (0.30 * evidence_strength) + \
                 (0.20 * signal_consistency) + \
                 (0.10 * profile_completeness)

    confidence = np.clip(confidence, 0.0, 1.0)

    if confidence > 0.70:
        conf_label = "High"
    elif confidence >= 0.40:
        conf_label = "Medium"
    else:
        conf_label = "Low"

    if yoe < target_yoe_min:
        exp_note = f"Warning: {yoe} YOE (below {target_yoe_min}+ target)."
    elif yoe > target_yoe_max + 2:
        exp_note = f"Note: {yoe} YOE (senior to target)."
    else:
        exp_note = f"Target YOE match ({yoe} years)."

    best_relative_idx = np.argmax(top_cand_scores)
    best_actual_idx = top_core_jd_indices[best_relative_idx]
    best_tag = UNIVERSAL_TECH_TAXONOMY[best_actual_idx].replace('_', ' ')

    worst_relative_idx = np.argmin(top_cand_scores)
    worst_actual_idx = top_core_jd_indices[worst_relative_idx]
    worst_tag = UNIVERSAL_TECH_TAXONOMY[worst_actual_idx].replace('_', ' ')

    raw_quote = evidence.get(UNIVERSAL_TECH_TAXONOMY[best_actual_idx], "NO_EVIDENCE")
    short_quote = textwrap.shorten(raw_quote, width=200, placeholder="...")

    best_tag_score = top_cand_scores[best_relative_idx]
    has_gap = cand_tax[worst_actual_idx] <= 0.50
    cid_bytes = str(row['candidate_id']).encode('utf-8')
    stable_hash = int(hashlib.md5(cid_bytes).hexdigest(), 16)
    variation = stable_hash % 5

    if best_tag_score < 0.65 or raw_quote == "NO_EVIDENCE":
        trapdoor_templates = [
            f"High semantic similarity to target role. {exp_note} While their overall background aligns with the JD vector, our strict evidence extractor could not isolate a definitive quote for {best_tag}. Recommend manual technical screen.",
            f"Strong overlap with broader retrieval and engineering systems. {exp_note} Note: We detected adjacent skills, but no direct quote proving explicit {best_tag} experience was extracted. Needs validation.",
            f"Broad alignment across multiple JD dimensions. {exp_note} However, concrete textual evidence for {best_tag} is missing from the parsed data. Human review advised.",
            f"Experience profile closely matches expected seniority. {exp_note} Ranked based on dense vector proximity, but we could not pinpoint an exact textual match for {best_tag}.",
            f"Profile appears adjacent to the core requirements. {exp_note} Caution: the system associates their background with {best_tag}, but lacks a definitive quote to verify direct implementation."
        ]
        reasoning = trapdoor_templates[variation]
    else:
        if rank <= 25:
            templates = [
                f"{exp_note} We rank this candidate highly due to their proven {best_tag} background: '{short_quote}'." + (f" They do show a slight gap in {worst_tag}, however." if has_gap else " No major weaknesses detected."),
                f"An exceptional match. Their work with {best_tag} stands out immediately ('{short_quote}')." + (f" Note a potential shortfall in {worst_tag}." if has_gap else "") + f" {exp_note}",
                f"'{short_quote}' — this explicitly validates their {best_tag} skills. {exp_note}" + (f" You may want to screen for {worst_tag} during the interview." if has_gap else ""),
                f"Strong profile across the board. {exp_note} They bring deep {best_tag} expertise, as noted by their claim: '{short_quote}'." + (f" Only minor concern is {worst_tag}." if has_gap else ""),
                f"Top-tier fit for the {best_tag} requirements ('{short_quote}')." + (f" We observed a lack of {worst_tag}." if has_gap else " Solid technical coverage.") + f" {exp_note}"
            ]
        elif rank <= 50:
            templates = [
                f"A very capable candidate. {exp_note} Their experience with {best_tag} is clear: '{short_quote}'." + (f" Keep in mind they are lighter on {worst_tag}." if has_gap else ""),
                f"{exp_note} This profile caught our attention specifically for {best_tag} ('{short_quote}')." + (f" Would need to verify their {worst_tag} knowledge." if has_gap else " Good overall alignment."),
                f"Solid secondary option. '{short_quote}' proves they can handle {best_tag}." + (f" The main drawback is their {worst_tag} exposure." if has_gap else "") + f" {exp_note}",
                f"Meets most core criteria, particularly {best_tag} ('{short_quote}'). {exp_note}" + (f" Missing strong signals for {worst_tag}." if has_gap else ""),
                f"Good overall background. {exp_note} They index highly on {best_tag}, stating '{short_quote}'." + (f" We didn't see much regarding {worst_tag}." if has_gap else "")
            ]
        elif rank <= 75:
            templates = [
                f"Borderline profile. While they highlight {best_tag} ('{short_quote}')," + (f" the lack of {worst_tag} lowers their rank." if has_gap else " their depth is average.") + f" {exp_note}",
                f"{exp_note} They have the required {best_tag} skills ('{short_quote}')." + (f" However, {worst_tag} is a noticeable blind spot." if has_gap else ""),
                f"Acceptable backup candidate. Their mention of '{short_quote}' checks the {best_tag} box." + (f" Fails the {worst_tag} checks, though." if has_gap else "") + f" {exp_note}",
                f"Ranked lower due to mixed signals. {exp_note} They do have {best_tag} ('{short_quote}')," + (f" but struggle with {worst_tag}." if has_gap else " but lack standout achievements."),
                f"Passable match. {exp_note} We verified {best_tag} via '{short_quote}'." + (f" Significant drop-off around {worst_tag}." if has_gap else "")
            ]
        else:
            templates = [
                f"Partial match. Shows some background in {best_tag} ('{short_quote}')." + (f" Disqualified for core roles based on {worst_tag} gaps." if has_gap else " Lacks strong primary signals.") + f" {exp_note}",
                f"Requires further review. {exp_note} While they trigger filters for {best_tag} ('{short_quote}')," + (f" they lack demonstrable {worst_tag} experience." if has_gap else " the broader profile is light on details."),
                f"{exp_note} Alternative profile. They possess baseline {best_tag} experience ('{short_quote}')." + (f" No direct evidence of {worst_tag}." if has_gap else ""),
                f"Secondary match. Highlights {best_tag} via '{short_quote}'." + (f" Does not meet the technical threshold for {worst_tag}." if has_gap else "") + f" {exp_note}",
                f"Atypical fit for this specific JD. {exp_note} Matches {best_tag} ('{short_quote}')" + (f" but is deeply deficient in {worst_tag}." if has_gap else " but offers limited alignment elsewhere.")
            ]

        reasoning = templates[variation]

    risk_flags = json.loads(row.get('risk_flags', '[]'))
    if risk_flags:
        flag_str = ", ".join(risk_flags)
        reasoning += f" [Note: Trust score adjusted due to: {flag_str}]."

    submission.append({
        "candidate_id": row['candidate_id'],
        "rank": rank,
        "score": round(row['final_score'], 4),
        "reasoning": reasoning
    })

output_df = pd.DataFrame(submission)
output_df = output_df[["candidate_id", "rank", "score", "reasoning"]]
output_df.to_csv("submission.csv", index=False)
print(f"[+] DONE. Validated 4-column submission.csv generated in {time.time() - start_time:.2f} seconds.")

[*] Loading offline universal artifacts...
[*] Loading LLM Taxonomy Weights & Parsing Raw JD...
[*] Dynamic YOE Target Extracted: 5 to 9 years
[*] Executing FAISS Exact Search...
[*] Purging Fatal Profiles to optimize compute...
[*] Remaining valid candidates for scoring: 4999
[*] Applying Hybrid Math & Behavioral Penalties...
[*] Sorting Top Validators...
[*] Generating Step 16 Confidence Scores and Reasoning...
[+] DONE. Validated 4-column submission.csv generated in 1.83 seconds.
